In [18]:
import os
import sys
import nibabel as nib
sys.path.append("../Classification")
import torch
from UNet import UNet3D
from save_nii import transform_nii
from monai.data import (Dataset, DataLoader)
from monai.inferers import sliding_window_inference
from monai.transforms import (Compose, ScaleIntensityRanged, NormalizeIntensityd, DeleteItemsd,
                             CastToTyped, SqueezeDimd, ToDeviced, LoadImaged, EnsureChannelFirstd, 
                              EnsureTyped, ToTensord)

In [19]:
import numpy as np

def compute_single_image_stats_percentile(img, pmin=0.5, pmax=99.5):
    img = img.astype(np.float32)
    vals = img[np.isfinite(img)]

    a_min, a_max = np.percentile(vals, [pmin, pmax])
    vals_clip = np.clip(vals, a_min, a_max)

    stats = {
        "min": float(a_min),
        "max": float(a_max),
        "mean": float(vals_clip.mean()),
        "std": float(vals_clip.std()),
        "range": float(a_max - a_min)
    }

    return stats


def preprocess_like_segmentation_transform_single_image(img, pmin=0.5, pmax=99.5):
    img = img.astype(np.float32)

    stats = compute_single_image_stats_percentile(img, pmin=pmin, pmax=pmax)

    a_min = stats["min"]
    a_max = stats["max"]
    img_range = stats["range"] + 1e-8

    # Equivalent to ScaleIntensityRanged
    img = np.clip(img, a_min, a_max)
    img = (img - a_min) / img_range

    # Equivalent to NormalizeIntensityd
    subtrahend = (stats["mean"] - stats["min"]) / img_range
    divisor = stats["std"] / img_range

    img = (img - subtrahend) / (divisor + 1e-8)

    return img.astype(np.float32), stats

In [20]:
path = "../DSApre/DSA154pre_img.nii.gz"
img = nib.load(path).get_fdata()[120:461, 120:461, 120:461]
#normalized_img = preprocess_like_segmentation_transform_single_image(img)[0]#[120:461, 120:461, 120:461]
#normalized_img = torch.tensor(normalized_img) 
transform_nii(torch.tensor(img), "DSA154pre_cropped_img.nii.gz")

In [5]:
cbct_path = "../BatchF/053-855122-G_img.nii.gz"
cbct_img = nib.load(cbct_path).get_fdata()

In [14]:
ts_files = [{"image": "DSA154pre_matched_img.nii.gz"}]

------

In [12]:
train_stats = {'mean': 2166.6000990825423, 'std': 2541.0911205286507, 'min': -1978.0, 'max': 17676.0, 'range': 19654.0}

test_transform = [
    LoadImaged(keys=["image"], ensure_channel_first=True, allow_missing_keys=True),
    ScaleIntensityRanged(
        keys="image",
        a_min=train_stats["min"],
        a_max=train_stats["max"],
        b_min=0,
        b_max=1,
        clip=True
    ),
    NormalizeIntensityd(keys="image",
                                subtrahend=(train_stats["mean"] - train_stats["min"]) / train_stats["range"],
                                divisor=train_stats["std"] / train_stats["range"]),
    EnsureTyped(keys=["image"], track_meta=False),
    DeleteItemsd(keys=['image_meta_dict']),
    ToTensord(keys=["image"], track_meta=False),
]

test_transforms = Compose(test_transform)

In [15]:
test_transforms = Compose(test_transform)
ts_ds = Dataset(data=ts_files, transform=test_transforms)
ts_loader = DataLoader(ts_ds, batch_size=1, shuffle=False, drop_last=False)

In [13]:
# Load the model
from UNet import UNet3D
model = UNet3D(in_channels=1, num_classes=5, strides=[1,2,2,2,2,2], channels=[16,32,64,128,256,512], prelu=True).cuda()
model = model.to('cuda')
model_data = torch.load('../Segmentation/epoch900_Proposed_small_lesion30_zeroshot_small_lr_ulb80.pth')
model.load_state_dict(model_data['model_state_dict'])

teacher_model = UNet3D(in_channels=1, num_classes=5, strides=[1,2,2,2,2,2], channels=[16,32,64,128,256,512], prelu=True).cuda()
teacher_model = teacher_model.to('cuda')
teacher_model.load_state_dict(model_data['teacher_model_state_dict'])

<All keys matched successfully>

In [16]:
def inference(model, test_loader, name):
    perf_dict = {}
    with torch.no_grad():
        dice = torch.zeros(5).cuda()
        for idx, batch in enumerate(test_loader):
            val_inputs = batch["image"].cuda()
            with torch.cuda.amp.autocast():
                val_seg_outputs = sliding_window_inference(
                    val_inputs, 
                    (128, 128, 128), 1,
                    model, 
                    device="cpu", 
                    progress=True, 
                    mode="constant", overlap=0.8
                )
                
            if name == "Proposed":
                val_seg_outputs = val_seg_outputs[0] 
            
            #plot_sdm_label(val_sdm_output, val_labels.cpu(), torch.argmax(val_seg_outputs, dim=1), 1)
            # Save the prediction to Nifti file
            filename = ts_files[idx]["image"].rpartition("_")[0]
            transform_nii(torch.argmax(val_seg_outputs, dim=1).squeeze(), 
                          "{}_pseudo.nii.gz".format(
                              filename)
                         )
            print(f"Pseudo Label for {filename} is Generated!")
            continue
            # Calculate the Dice for segmentation task
            val_labels_list = decollate_batch(val_labels)
            val_labels_convert = [post_label(val_label_tensor) for val_label_tensor in val_labels_list]
            val_outputs_list = decollate_batch(val_seg_outputs)
            val_output_convert = [post_pred(val_pred_tensor).cuda() for val_pred_tensor in val_outputs_list]
            dice_metric(y_pred=val_output_convert, y=val_labels_convert)
            print(dice_metric(y_pred=val_output_convert, y=val_labels_convert))
            perf_dict[test_files[idx]["label"].split('/')[-1].split('.')[0].rsplit("-SEG")[0]] = dice_metric(y_pred=val_output_convert, y=val_labels_convert).squeeze().tolist()
            dice += dice_metric(y_pred=val_output_convert, y=val_labels_convert).view(-1)
            print("Completed")
        print(dice/len(test_loader))
        print("Complete!")
        return perf_dict

In [17]:
inference(teacher_model, ts_loader, "Proposed")

100%|██████████| 1000/1000 [00:46<00:00, 21.72it/s]


Pseudo Label for DSA154pre_matched is Generated!
tensor([0., 0., 0., 0., 0.], device='cuda:0')
Complete!


{}

In [6]:
import numpy as np
from skimage.exposure import match_histograms

def histogram_match_for_segmentation(
    source_img,
    reference_img,
    pmin=0.5,
    pmax=99.5,
    mask_source=None,
    mask_reference=None
):
    source_img = source_img.astype(np.float32)
    reference_img = reference_img.astype(np.float32)

    # optional: foreground / valid-region 기반으로 clipping range 계산
    if mask_source is not None:
        source_vals = source_img[mask_source > 0]
    else:
        source_vals = source_img[np.isfinite(source_img)]

    if mask_reference is not None:
        ref_vals = reference_img[mask_reference > 0]
    else:
        ref_vals = reference_img[np.isfinite(reference_img)]

    src_lo, src_hi = np.percentile(source_vals, [pmin, pmax])
    ref_lo, ref_hi = np.percentile(ref_vals, [pmin, pmax])

    source_clip = np.clip(source_img, src_lo, src_hi)
    reference_clip = np.clip(reference_img, ref_lo, ref_hi)

    # histogram matching
    matched = match_histograms(
        source_clip,
        reference_clip,
        channel_axis=None
    )

    # reference domain range로 clip
    matched = np.clip(matched, ref_lo, ref_hi)

    return matched.astype(np.float32)

In [7]:
matched_img = histogram_match_for_segmentation(
    source_img=img,
    reference_img=cbct_img,
    pmin=0.5,
    pmax=99.5
)